# Surveying: traverse adjustment and levelling

> **Disclaimer.** CivKit results support, and do not replace, the judgement of a qualified engineer.
>
> Everything here runs offline: the `civkit` package makes no network calls.

The examples are the published cases CivKit is tested against (US Navy, *Engineering Aid*, NAVEDTRA 14070 and 14069;
public domain). See `tests/reference/` for the full case files.

In [ ]:
import civkit

## Closed traverse, compass (Bowditch) rule

Quadrant bearings are given with the unit `bearing`; distances in feet. The result has latitudes, departures, the
misclosure and the adjusted values, one per leg.

In [ ]:
t = civkit.surveying.traverse.compass_rule(
    bearings=(["N 3d45m E", "S 86d15m E", "S 10d49m E", "S 32d40m W", "N 43d29m W", "S 57d31m W"], "bearing"),
    distances=([584.21, 720.26, 212.0, 292.31, 278.53, 527.54], "ft"),
    output_units={name: "ft" for name in ["latitudes", "departures", "adjusted_latitudes", "adjusted_departures",
                                          "latitude_misclosure", "departure_misclosure", "linear_misclosure", "perimeter"]},
)
print(f"{'leg':>3} {'latitude':>10} {'departure':>10} {'adj. lat.':>10} {'adj. dep.':>10}")
o = t.outputs
for k in range(len(o["latitudes"])):
    print(f"{k + 1:>3} {o['latitudes'][k].value:10.2f} {o['departures'][k].value:10.2f} "
          f"{o['adjusted_latitudes'][k].value:10.2f} {o['adjusted_departures'][k].value:10.2f}")
print("linear misclosure", round(o["linear_misclosure"].value, 4), o["linear_misclosure"].unit,
      "; relative", f"1 : {1 / o['relative_misclosure'].value:.0f}")

## Differential levelling: two reductions, one answer

The height-of-instrument and rise-and-fall methods reduce the same field book to the same elevations.

In [ ]:
book = dict(
    start_elevation=(133.163, "ft"),
    backsights=([6.659, 4.968, 4.508, 1.412, 7.073], "ft"),
    foresights=([4.971, 7.452, 5.857, 1.261, 1.785], "ft"),
    output_units={"elevations": "ft"},
)
hi = civkit.surveying.levelling.height_of_instrument(**book)
rf = civkit.surveying.levelling.rise_and_fall(**book)
for k, (a, b) in enumerate(zip(hi.outputs["elevations"], rf.outputs["elevations"]), start=1):
    print(f"point {k}: {a.value:.3f} ft (HI)   {b.value:.3f} ft (rise and fall)")
    assert abs(a.value - b.value) < 1e-9

## Closing on a benchmark: closure error and proportional adjustment

The run closes on a benchmark of known elevation; the misclosure is distributed in proportion to distance.

In [ ]:
c = civkit.surveying.levelling.closure_adjustment(
    elevations=([134.851, 132.367, 131.018, 131.169, 136.457], "ft"),
    section_lengths=([440, 500, 620, 200, 380], "ft"),
    closing_elevation=(136.442, "ft"),
    output_units={"closure_error": "ft", "total_length": "ft", "adjusted_elevations": "ft"},
)
print(f"closure error {c.outputs['closure_error'].value:.3f} ft over {c.outputs['total_length'].value:.3f} ft")
for k, q in enumerate(c.outputs["adjusted_elevations"], start=1):
    print(f"point {k}: {q.value:.3f} ft")